In [9]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/ieee-fraud-detection/sample_submission.csv
/kaggle/input/ieee-fraud-detection/test_identity.csv
/kaggle/input/ieee-fraud-detection/train_identity.csv
/kaggle/input/ieee-fraud-detection/test_transaction.csv
/kaggle/input/ieee-fraud-detection/train_transaction.csv


In [10]:
!pip install mlflow dagshub --quiet
import mlflow
from dagshub import dagshub_logger
import os

# Set tracking URI manually
mlflow.set_tracking_uri("https://dagshub.com/ekvirika/FraudDerection.mlflow")

# Use your DagsHub credentials
os.environ["MLFLOW_TRACKING_USERNAME"] = "ekvirika"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "3f601f2c2c7a6bca448ebc69f4f5d4b49daffd8f"

# Optional: set registry if you're using model registry
mlflow.set_registry_uri("https://dagshub.com/ekvirika/FraudDerection.mlflow")

In [11]:
!pip install imbalanced-learn==0.11.0 --quiet
import os
import matplotlib.pylab as plt
import warnings
warnings.filterwarnings('ignore')
import altair as alt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from scipy import sparse
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import RFE
from lightgbm import LGBMClassifier
from scipy import stats
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve, auc
from xgboost import XGBClassifier
import datetime
from datetime import timedelta
from imblearn.over_sampling import SMOTE

In [12]:
pd.set_option('display.max_columns', None)  
pd.set_option('display.width', None)        
pd.set_option('display.expand_frame_repr', False)

In [13]:
# Input data files path
folder_path = '/kaggle/input/ieee-fraud-detection/'

# Load training data
train_identity = pd.read_csv(f'{folder_path}train_identity.csv')
train_transaction = pd.read_csv(f'{folder_path}train_transaction.csv')

# Merge the training data
train = pd.merge(train_transaction, train_identity, on='TransactionID', how='left')
del train_identity, train_transaction

# Feature Engineering

# Cleaning and Feature Engineering pipeline classes

In [14]:
class DropNullValues(BaseEstimator, TransformerMixin):
    """
    Drops columns based on null threshold and value dominance
    """
    def __init__(self, null_threshold=0.9, value_dominance_threshold=0.9):
        self.null_threshold = null_threshold
        self.value_dominance_threshold = value_dominance_threshold
        self.cols_to_drop_ = None
    
    def fit(self, X, y=None):
        many_null_cols = self._get_null_columns(X)
        big_top_value_cols = self._get_dominant_value_columns(X)
        one_value_cols = self._get_single_value_columns(X)
        
        self.cols_to_drop_ = list(set(many_null_cols + big_top_value_cols + one_value_cols))
        
        self.cols_to_drop_ = [col for col in self.cols_to_drop_ if col != 'isFraud']
        
        return self
    
    def transform(self, X):
        return X.drop(columns=self.cols_to_drop_, axis=1, errors='ignore')
    
    def _get_null_columns(self, df):
        return [col for col in df.columns 
                if df[col].isnull().sum() / len(df) > self.null_threshold]
    
    def _get_dominant_value_columns(self, df):
        dominant_cols = []
        for col in df.columns:
            value_counts = df[col].value_counts(dropna=False, normalize=True)
            if len(value_counts) > 0 and value_counts.iloc[0] > self.value_dominance_threshold:
                dominant_cols.append(col)
        return dominant_cols
    
    def _get_single_value_columns(self, df):
        return [col for col in df.columns if df[col].nunique(dropna=False) == 1]

In [15]:
class CorrelationFilter(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.95):
        self.threshold = threshold
        self.columns_to_drop_ = []

    def fit(self, X, y=None):
        # Convert to DataFrame if it's a NumPy array
        if isinstance(X, np.ndarray):
            X = pd.DataFrame(X)

        corr_matrix = X.corr().abs()
        upper = np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
        upper_tri = corr_matrix.where(upper)

        self.columns_to_drop_ = [column for column in upper_tri.columns if any(upper_tri[column] > self.threshold)]
        return self

    def transform(self, X):
        # Convert to DataFrame if it's a NumPy array
        if isinstance(X, np.ndarray):
            X = pd.DataFrame(X)

        return X.drop(columns=self.columns_to_drop_, errors='ignore')


In [16]:
class MissingValueImputer(BaseEstimator, TransformerMixin):
    """
    Imputes missing values for categorical and numerical columns
    """
    def __init__(self, cat_fill_value="Unknown", num_strategy="median"):
        self.cat_fill_value = cat_fill_value
        self.num_strategy = num_strategy
        self.num_fill_values_ = {}
        self.categorical_columns_ = None
        self.numerical_columns_ = None
    
    def fit(self, X, y=None):
        self.categorical_columns_ = X.select_dtypes(include=['object', 'category']).columns.tolist()
        self.numerical_columns_ = X.select_dtypes(include=['number']).columns.tolist()
        
        if self.num_strategy == "median":
            for col in self.numerical_columns_:
                self.num_fill_values_[col] = X[col].median()
        
        return self
    
    def transform(self, X):
        X_transformed = X.copy()
        
        for col in self.categorical_columns_:
            X_transformed[col] = X_transformed[col].fillna(self.cat_fill_value)
        
        for col in self.numerical_columns_:
            X_transformed[col] = X_transformed[col].fillna(self.num_fill_values_[col])
        
        return X_transformed

In [17]:
class CategoricalEncoder(BaseEstimator, TransformerMixin):
    """
    Encodes categorical variables using Weight of Evidence
    """
    def __init__(self, threshold=3):
        self.threshold = threshold
    
    def fit(self, X, y):
        cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
        s = X[cat_cols].nunique()
        self.woe_columns = s[s > self.threshold].index.tolist()
        self.one_hot_columns = s[s <= self.threshold].index.tolist()
        
        # Store most frequent category to handle missing values
        self.woe_columns_fill_na = X[self.woe_columns].mode().iloc[0].to_dict()
        
        woe_mappings = {}
        iv_values = {}
        df = X.copy()
        df['_target_'] = y
        
        for col in self.woe_columns:
            groups = df.groupby(col, dropna=False)['_target_'].agg(['count', 'sum'])
            groups.columns = ['n_obs', 'n_pos']
            groups['n_neg'] = groups['n_obs'] - groups['n_pos']
            groups['prop_pos'] = groups['n_pos'] / max(groups['n_pos'].sum(), 1e-6)
            groups['prop_neg'] = groups['n_neg'] / max(groups['n_neg'].sum(), 1e-6)
            groups['woe'] = np.log((groups['prop_pos'] + 1e-6) / (groups['prop_neg'] + 1e-6))
            groups['iv'] = (groups['prop_pos'] - groups['prop_neg']) * groups['woe']
            groups = groups.replace([np.inf, -np.inf], 0).fillna(0)
            woe_mappings[col] = groups['woe'].to_dict()
            iv_values[col] = groups['iv'].sum()
            
        self.woe_mappings = woe_mappings
        self.iv_values = iv_values
        return self
    
    def transform(self, X):
        X_transformed = X.copy()
        for col in self.woe_columns:
            mapping = self.woe_mappings[col]
            default_value = mapping.get(self.woe_columns_fill_na[col], 0)
            
            # Transform column and handle unseen values
            transformed_col = X_transformed[col].map(lambda x: mapping.get(x, default_value))
            X_transformed[f'woe_{col}'] = transformed_col
            X_transformed.drop(columns=col, inplace=True)
        
        # One-hot encode remaining columns
        if self.one_hot_columns:
            X_transformed = pd.get_dummies(
                X_transformed,
                columns=self.one_hot_columns,
                drop_first=True,
                dummy_na=True,
                dtype=int
            )
        
        # Final check for NaNs
        if X_transformed.isnull().any().any():
            nan_cols = X_transformed.columns[X_transformed.isnull().any()].tolist()
            raise ValueError(f"Unexpected NaNs found in columns: {nan_cols}")
            
        return X_transformed

In [18]:
class NumericalProcessor(BaseEstimator, TransformerMixin):
    """
    Identifies outliers in numerical columns using modified Z-score
    """
    def __init__(self, z_threshold=3.5):
        self.z_threshold = z_threshold
        self.medians_ = {}
        self.mads_ = {}

    def fit(self, X, y=None):
        print('NumericalProcessor\n')
        num_cols = X.select_dtypes(include=np.number).columns
        self.medians_ = {col: X[col].median() for col in num_cols}
        self.mads_ = {col: np.median(np.abs(X[col] - self.medians_[col])) + 1e-10 for col in num_cols}
        return self

    def transform(self, X):
        X = X.copy()
        for col in self.medians_:
            med = self.medians_[col]
            mad = self.mads_[col]
            modified_z = 0.6745 * (X[col] - med) / mad  # Modified Z-score
            X[f'{col}_outlier'] = (np.abs(modified_z) > self.z_threshold).astype(int)
        return X

In [19]:
class TimeFeatureExtractor(BaseEstimator, TransformerMixin):
    """
    Extracts time-related features from TransactionDT
    """
    def __init__(self, start_date='2017-12-01'):
        self.start_date = datetime.datetime.strptime(start_date, '%Y-%m-%d')
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X_transformed = X.copy()
        
        # Convert TransactionDT to datetime
        X_transformed['datetime'] = X_transformed['TransactionDT'].apply(
            lambda x: self.start_date + timedelta(seconds=x)
        )
        
        # Extract time features
        X_transformed['hour'] = X_transformed['datetime'].dt.hour
        X_transformed['day'] = X_transformed['datetime'].dt.day
        X_transformed['weekday'] = X_transformed['datetime'].dt.weekday
        X_transformed['month'] = X_transformed['datetime'].dt.month
        X_transformed['year'] = X_transformed['datetime'].dt.year
        
        # Time periods
        X_transformed['is_weekend'] = X_transformed['weekday'].apply(lambda x: 1 if x >= 5 else 0)
        X_transformed['is_night'] = X_transformed['hour'].apply(lambda x: 1 if (x >= 22 or x <= 5) else 0)
        X_transformed['is_morning'] = X_transformed['hour'].apply(lambda x: 1 if (6 <= x <= 11) else 0)
        X_transformed['is_afternoon'] = X_transformed['hour'].apply(lambda x: 1 if (12 <= x <= 17) else 0)
        X_transformed['is_evening'] = X_transformed['hour'].apply(lambda x: 1 if (18 <= x <= 21) else 0)
        
        # Drop original datetime
        X_transformed.drop('datetime', axis=1, inplace=True)
        
        return X_transformed

In [20]:
class UserBehaviorFeatureExtractor(BaseEstimator, TransformerMixin):
    """
    Extracts user behavior features based on user_id (card1, card2, addr1)
    """
    def __init__(self, window_sizes=[1, 7, 30]):
        self.window_sizes = window_sizes
        self.user_stats = {}
    
    def fit(self, X, y=None):
        if y is None:
            return self
            
        df = X.copy()
        df['isFraud'] = y
        
        # Create a user_id combining card1, card2, and addr1 - key identifiers
        df['user_id'] = df['card1'].astype(str) + '_' + df['card2'].astype(str) + '_' + df['addr1'].astype(str)
        
        # Calculate overall user statistics
        user_stats = df.groupby('user_id').agg({
            'TransactionID': 'count',  # Number of transactions
            'isFraud': ['mean', 'sum']  # Fraud rate and total
        })
        
        user_stats.columns = ['tx_count', 'fraud_rate', 'fraud_count']
        self.user_stats = user_stats
        
        return self
    
    def transform(self, X):
        X_transformed = X.copy()
        
        # Create user_id
        X_transformed['user_id'] = X_transformed['card1'].astype(str) + '_' + X_transformed['card2'].astype(str) + '_' + X_transformed['addr1'].astype(str)
        
        # Add user behavior features
        X_transformed['user_tx_count'] = X_transformed['user_id'].map(self.user_stats['tx_count']).fillna(1)
        X_transformed['user_fraud_rate'] = X_transformed['user_id'].map(self.user_stats['fraud_rate']).fillna(0)
        X_transformed['user_fraud_count'] = X_transformed['user_id'].map(self.user_stats['fraud_count']).fillna(0)
        
        # Feature: Average transaction amount per user
        avg_amount = X_transformed.groupby('user_id')['TransactionAmt'].mean().to_dict()
        X_transformed['user_avg_amount'] = X_transformed['user_id'].map(avg_amount).fillna(X_transformed['TransactionAmt'])
        
        # Feature: Transaction amount vs user average
        X_transformed['amount_vs_user_avg'] = X_transformed['TransactionAmt'] / X_transformed['user_avg_amount']
        
        # Feature: Velocity - transactions per day per user
        if 'day' in X_transformed.columns:
            tx_per_day = X_transformed.groupby(['user_id', 'day']).size().groupby('user_id').mean().to_dict()
            X_transformed['user_tx_per_day'] = X_transformed['user_id'].map(tx_per_day).fillna(1)
        
        # Drop user_id as it's no longer needed
        X_transformed.drop('user_id', axis=1, inplace=True)
        
        return X_transformed

In [21]:
class SelectColumn(BaseEstimator, TransformerMixin):
    """
    Selects specific columns from the dataset
    """
    def __init__(self, columns):
        self.columns = columns

    def fit(self, X, y=None):
        return self  

    def transform(self, X):
        # Only select columns that actually exist in X
        valid_columns = [col for col in self.columns if col in X.columns]
        return X[valid_columns]

# Final Pipeline

In [22]:
# Create the full preprocessing pipeline
preprocessing = Pipeline([
    ('drop_na', DropNullValues()),
    ('imputer', MissingValueImputer()),
    # ('correlation', CorrelationFilter(threshold=0.9)),
    ('time_extractor', TimeFeatureExtractor()),
    ('user_behavior_extractor', UserBehaviorFeatureExtractor()),
    ('cat_encoder', CategoricalEncoder()),
    ('num_processor', NumericalProcessor()),
    ('scaler', StandardScaler())
])

# Training

In [23]:
# Prepare the data
X = train.drop('isFraud', axis=1)
y = train['isFraud']
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Dimensions of the train dataset is {X_train.shape} and {y_train.shape}')
print(f'Dimensions of the validation dataset is {X_valid.shape} and {y_valid.shape}')

# Process the datasets
print("Starting data preprocessing...")
import time


start_time = time.time()
X_t = preprocessing.fit_transform(X_train, y_train)
end_time = time.time()
elapsed_time = end_time - start_time
print(f"Data preprocessing completed in {elapsed_time:.2f} seconds")

X_v = preprocessing.transform(X_valid)

Dimensions of the train dataset is (472432, 433) and (472432,)
Dimensions of the validation dataset is (118108, 433) and (118108,)
Starting data preprocessing...
NumericalProcessor

Data preprocessing completed in 55.51 seconds


# Feature Selection using RFE

In [ ]:
# Now let's select top features with RFE
print("Starting feature selection using RFE...")
num_features = 231  # You can change this value for different number of features

xgb_model = XGBClassifier(n_jobs=-1, eval_metric="auc", max_depth=25, min_child_weight=1, reg_alpha=0, reg_lambda=10)
rfe = RFE(estimator=xgb_model, n_features_to_select=num_features, step=0.1)
rfe.fit(X_t, y_train)

# Get the selected features
X_train_selected = rfe.transform(X_t)
X_valid_selected = rfe.transform(X_v)
selected_features = X_t.columns[rfe.support_].tolist()

print(f"Selected {len(selected_features)} features")

# Build final pipeline
preprocess_and_selecting = Pipeline([
    ('preprocessing', preprocessing),
    ('selector', SelectColumn(columns=selected_features)),
])

# Process datasets with the final pipeline
X_train_processed = preprocess_and_selecting.fit_transform(X_train, y_train)
X_valid_processed = preprocess_and_selecting.transform(X_valid)

# Train and evaluate the XGBoost model
print("Training XGBoost model...")
xgb = XGBClassifier(n_jobs=-1, eval_metric="auc", max_depth=25, min_child_weight=1, reg_alpha=0, reg_lambda=10)
xgb.fit(X_train_processed, y_train)

Starting feature selection using RFE...


In [ ]:
# Make predictions and calculate AUC
y_predict = xgb.predict_proba(X_valid_processed)[:, 1]
roc_auc = roc_auc_score(y_valid, y_predict)
print(f"XGBoost model ROC AUC: {roc_auc:.4f}")

# Visualize the ROC curve
plt.figure(figsize=(8, 6))
fpr, tpr, _ = roc_curve(y_valid, y_predict)
plt.plot(fpr, tpr, label=f'XGBoost (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve for IEEE Fraud Detection')
plt.legend()
plt.grid(True)
plt.savefig("roc_curve.png")
plt.show()

# Create a complete pipeline
pipeline = Pipeline([
    ('preprocess_select', preprocess_and_selecting),
    ('model', xgb)
])


pipeline.fit(X_train, y_train)


# Mlflow logging

In [ ]:
import mlflow
import pandas as pd 
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
import numpy as np

# Set the experiment name
mlflow.set_experiment("XGBoost_Training")

with mlflow.start_run(run_name='XGboost_Model_Enhanced'):
    # Log the model
    mlflow.sklearn.log_model(pipeline, "xgboost_model")
    
    # Get predictions and calculate AUC
    y_train_proba = pipeline.predict_proba(X_train)[:, 1]
    y_valid_proba = pipeline.predict_proba(X_valid)[:, 1]
    
    # Fix: Correct ROC curve calculation
    fpr_train, tpr_train, _ = roc_curve(y_train, y_train_proba)
    fpr_valid, tpr_valid, _ = roc_curve(y_valid, y_valid_proba)
    auc_train = auc(fpr_train, tpr_train)
    auc_valid = auc(fpr_valid, tpr_valid)
    
    # Log metrics
    mlflow.log_metric("train_auc", auc_train)
    mlflow.log_metric("valid_auc", auc_valid)
    
    # Plot and log ROC curve
    plt.figure(figsize=(8, 6))
    plt.plot(fpr_train, tpr_train, color='red', label=f'Train ROC (AUC = {auc_train:.4f})')
    plt.plot(fpr_valid, tpr_valid, color='blue', label=f'Validation ROC (AUC = {auc_valid:.4f})')
    plt.plot([0, 1], [0, 1], 'k--', label='Random (AUC = 0.50)')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve - Train vs Validation')
    plt.legend(loc='lower right')
    plt.grid(True)
    roc_image_path = "roc_curve_comparison.png"
    plt.savefig(roc_image_path)
    mlflow.log_artifact(roc_image_path)
    plt.close()
    
    # Plot and log probability distribution
    plt.figure(figsize=(8, 6))
    plt.hist(y_valid_proba, bins=50, alpha=0.75, label="Validation set")
    plt.hist(y_train_proba, bins=50, alpha=0.75, label="Train set")
    plt.title("Predicted Probability Distribution")
    plt.xlabel("Predicted Probability")
    plt.ylabel("Frequency")
    plt.legend(loc="upper right")
    prob_dist_image_path = "probability_distribution.png"
    plt.savefig(prob_dist_image_path)
    mlflow.log_artifact(prob_dist_image_path)
    plt.close()
    
    # Get the XGBoost model from the pipeline
    # Assuming the XGBoost model is the last step in your pipeline
    xgb_model = pipeline.named_steps['model']  # Adjust this based on your pipeline's structure
    
    # Extract feature importance
    feature_importance = pd.DataFrame({
        'Feature': selected_features,
        'Importance': xgb_model.feature_importances_
    }).sort_values('Importance', ascending=False)
    
    # Save and log feature importance as CSV
    feature_importance.to_csv("feature_importance.csv", index=False)
    mlflow.log_artifact("feature_importance.csv")
    
    # Plot and log feature importance
    plt.figure(figsize=(10, 8))
    plt.barh(feature_importance['Feature'][:20], feature_importance['Importance'][:20])
    plt.xlabel('Importance')
    plt.title('Top 20 Feature Importance')
    plt.tight_layout()
    feature_imp_path = "feature_importance.png"
    plt.savefig(feature_imp_path)
    mlflow.log_artifact(feature_imp_path)
    plt.close()
    
    # Log parameters
    mlflow.log_param("num_features_selected", len(selected_features))
    mlflow.log_param("xgboost_max_depth", xgb_model.max_depth)
    mlflow.log_param("xgboost_min_child_weight", xgb_model.min_child_weight)
    mlflow.log_param("xgboost_reg_alpha", xgb_model.reg_alpha)
    mlflow.log_param("xgboost_reg_lambda", xgb_model.reg_lambda)
    mlflow.log_param("time_features_extracted", True)
    mlflow.log_param("user_behavior_features_extracted", True)
    
    # Get the run ID for model registration
    run_id = mlflow.active_run().info.run_id
    
    # Register the model in the MLflow Model Registry
    model_uri = f"runs:/{run_id}/xgboost_model"
    model_name = "XGBoost_Classification_Model"
    model_version = mlflow.register_model(model_uri, model_name)
    
    print(f"Model registered with name: {model_name}")
    print(f"Model version: {model_version.version}")
    
    # Optionally: Set model version stage (e.g., "Staging", "Production")
    client = mlflow.tracking.MlflowClient()
    client.transition_model_version_stage(
        name=model_name,
        version=model_version.version,
        stage="Staging"
    )
    
    # Add model description and tags
    client.update_model_version(
        name=model_name,
        version=model_version.version,
        description=f"XGBoost classifier with validation AUC: {auc_valid:.4f}"
    )
    
    # Log model signature and input example (optional but recommended)
    # This helps with model deployment later
    from mlflow.models.signature import infer_signature
    
    signature = infer_signature(X_valid, y_valid_proba)
    example = X_valid.iloc[:5]
    
    mlflow.sklearn.log_model(
        pipeline, 
        "xgboost_model_with_signature",
        signature=signature,
        input_example=example
    )